# Stage 3 — Action Values (VAEP) with Leave-One-League-Out Validation

**Pipeline position:** `data/stage1_stage2_features.csv` → **Stage 3** → `stage3_vaep/vaep_all_leagues.parquet` → Stage 5

Two gradient-boosted classifiers (CatBoost) estimate, for every game state, the probability that the acting team
**scores** and **concedes** within the next ten actions. Each league is predicted by models trained on the other
four (leave-one-league-out), so no action is valued by a model that saw its match.

An action's value is the change it causes in the two probabilities:

```
V(aᵢ) = ΔP_scores(aᵢ) − ΔP_concedes(aᵢ)
```

Stage 5 keeps the two changes apart and weights each by what the next goal is worth.

**Runtime:** about 10 minutes per fold on a T4 GPU (5 folds, 10 models); see the README for hardware.

**Outputs (`stage3_vaep/`)**

| File | Content |
|---|---|
| `models/fold_<League>_{scores,concedes}.cbm` | the 10 trained models |
| `predictions/vaep_predictions_<League>.parquet` | per-action predictions for each held-out league |
| `vaep_all_leagues.parquet` | all 2.83M actions combined — input to Stage 5 |
| `evaluation_metrics.csv`, `model_training_info.csv`, `plots/` | out-of-league Brier score and ROC-AUC, calibration |

## 0. Setup

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q catboost pandas numpy scikit-learn matplotlib seaborn tqdm pyarrow

In [ ]:
import os, gc, warnings, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
sns.set_style("whitegrid")

# GPU check
import subprocess
try:
    gpu_info = subprocess.check_output("nvidia-smi", shell=True).decode()
    print(gpu_info[:500])
    HAS_GPU = True
except Exception:
    print("⚠️  No GPU detected — CatBoost will fall back to CPU (slower).")
    HAS_GPU = False

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Repository root (the notebooks live one level below it)
PROJECT_ROOT = Path('..') if Path.cwd().name in ('pipeline-stages', 'feature-engineering') else Path('.')
CSV_PATH     = PROJECT_ROOT / "data/stage1_stage2_features.csv"

STAGE3_DIR = PROJECT_ROOT / "stage3_vaep"
MODELS_DIR = STAGE3_DIR / "models"
PREDS_DIR  = STAGE3_DIR / "predictions"
PLOTS_DIR  = STAGE3_DIR / "plots"
for d in [STAGE3_DIR, MODELS_DIR, PREDS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Input:  {CSV_PATH}  (exists: {CSV_PATH.exists()})")
print(f"Output: {STAGE3_DIR}")

## 1. Load Data

In [ ]:
# ── Columns ───────────────────────────────────────────────────────────────────
ID_COLS = [
    "game_id", "league_country", "period_id", "absolute_second",
    "team_id", "player_id", "minute",
]

VAEP_FEATURES = [
    # Current action
    "type_id", "result_id", "bodypart_id",
    "start_x", "start_y", "end_x", "end_y",
    # Previous action (a-1)
    "type_id_a1", "result_id_a1", "bodypart_id_a1",
    "start_x_a1", "start_y_a1", "end_x_a1", "end_y_a1",
    # Two actions back (a-2)
    "type_id_a2", "result_id_a2", "bodypart_id_a2",
    "start_x_a2", "start_y_a2", "end_x_a2", "end_y_a2",
    # Geometric
    "dist_to_goal_start", "angle_to_goal_start",
    "dist_to_goal_end", "angle_to_goal_end",
    "dx", "dy", "action_distance",
    "time_since_prev", "dist_from_prev", "possession_change",
    # Game context
    "goals_for", "goals_against", "goal_difference",
    "score_margin", "score_state",
    # Late-game
    "minutes_remaining",
    "is_substitute", "minutes_on_pitch",
]

EXTRA_COLS = ["is_post_75", "action_type", "result"]

USE_COLS = list(dict.fromkeys(ID_COLS + VAEP_FEATURES + EXTRA_COLS))

CATEGORICAL_FEATURES = [
    "type_id", "result_id", "bodypart_id",
    "type_id_a1", "result_id_a1", "bodypart_id_a1",
    "type_id_a2", "result_id_a2", "bodypart_id_a2",
    "score_state",
]

print(f"Loading {len(USE_COLS)} columns...")

In [ ]:
t0 = time.time()
df = pd.read_csv(CSV_PATH, usecols=USE_COLS, low_memory=False)
print(f"Loaded {len(df):,} rows × {df.shape[1]} cols in {time.time()-t0:.1f}s")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e6:.0f} MB")
print(df["league_country"].value_counts())

In [ ]:
# ── Cleaning ──────────────────────────────────────────────────────────────────
# 1. Rolling-window nulls (first 1–2 actions per game)
a1a2 = [c for c in df.columns if c.endswith(("_a1", "_a2"))]
df[a1a2] = df[a1a2].fillna(0)

# 2. First-action-of-game nulls for time_since_prev / dist_from_prev
df["time_since_prev"] = df["time_since_prev"].fillna(0).clip(lower=0)
df["dist_from_prev"]  = df["dist_from_prev"].fillna(0)

# 3. Encode score_state as integer
score_state_map = {"chasing": 0, "level": 1, "protecting": 2}
df["score_state"] = df["score_state"].map(score_state_map).fillna(1).astype(int)

# 4. Ensure categoricals are int
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype(int)

# Verify
nulls = df[VAEP_FEATURES].isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print("✅ Zero nulls in feature columns")
else:
    print(f"⚠️  Remaining nulls:\n{nulls}")

## 2. Labels (k = 10)

For every action `aᵢ` two binary labels are built over the window `{aᵢ, aᵢ₊₁, …, aᵢ₊₉}`:

- **`y_scores`** — the acting team scores within the window
- **`y_concedes`** — the acting team concedes within the window

The window includes the action itself, so a goal-scoring shot is labelled as a goal (as in the reference
implementation of Decroos et al., 2019). Labels are cached to `vaep_labels.parquet`.

In [ ]:
LABELS_PATH = STAGE3_DIR / "vaep_labels.parquet"
K = 10
SHOT_TYPES = {"shot", "shot_penalty", "shot_freekick"}

goal_mask = df["action_type"].isin(SHOT_TYPES) & (df["result"] == "success")
df["is_goal_action"] = goal_mask.values          # used by the value function (goal reset)

if LABELS_PATH.exists():
    print(f"Loading cached labels from {LABELS_PATH}")
    labels_df = pd.read_parquet(LABELS_PATH)
    assert len(labels_df) == len(df), "cached labels do not match df length"
    df["y_scores"]   = labels_df["y_scores"].values
    df["y_concedes"] = labels_df["y_concedes"].values
    del labels_df
    gc.collect()
else:
    def construct_vaep_labels(game_df):
        """Window = {i} u {i+1 .. i+K-1}, matching socceraction."""
        n   = len(game_df)
        tid = game_df["team_id"].values
        goal = game_df["is_goal_action"].values.astype(bool)

        y_scores   = goal.copy()                      # j == i  (res = y["goal"])
        y_concedes = np.zeros(n, dtype=bool)          # owngoals absent in this data

        for k in range(1, K):                         # j = i+1 .. i+K-1
            g_sh = np.zeros(n, dtype=bool)
            g_sh[: n - k] = goal[k:]                  # pad tail with False
            t_sh = np.empty(n, dtype=tid.dtype)
            t_sh[: n - k] = tid[k:]
            t_sh[n - k :] = tid[-1]

            y_scores   |= g_sh & (t_sh == tid)
            y_concedes |= g_sh & (t_sh != tid)

        return y_scores.astype(np.int8), y_concedes.astype(np.int8)

    y_s = np.zeros(len(df), dtype=np.int8)
    y_c = np.zeros(len(df), dtype=np.int8)

    for gid, idx in tqdm(df.groupby("game_id", sort=False).indices.items(),
                         total=df["game_id"].nunique(), desc="Building labels"):
        ys, yc = construct_vaep_labels(df.iloc[idx])
        y_s[idx] = ys
        y_c[idx] = yc

    df["y_scores"]   = y_s
    df["y_concedes"] = y_c

    df[["game_id", "y_scores", "y_concedes"]].to_parquet(LABELS_PATH, index=True)
    print(f"Labels saved -> {LABELS_PATH}")

print(f"y_scores   rate: {df['y_scores'].mean():.4f}  ({df['y_scores'].sum():,})")
print(f"y_concedes rate: {df['y_concedes'].mean():.4f}  ({df['y_concedes'].sum():,})")
print(f"Reference (Decroos et al. 2019): ~0.015 / ~0.005")

## 3. Leave-One-League-Out Folds

In [ ]:
LEAGUES = sorted(df["league_country"].unique().tolist())

LOLO_FOLDS = {
    held_out: [l for l in LEAGUES if l != held_out]
    for held_out in LEAGUES
}

for held_out, train_leagues in LOLO_FOLDS.items():
    n_test  = (df["league_country"] == held_out).sum()
    n_train = df["league_country"].isin(train_leagues).sum()
    print(f"Fold {held_out:10s}:  train {n_train:>9,}  |  test {n_test:>9,}  |  train leagues: {train_leagues}")

# Categorical feature indices within VAEP_FEATURES
CAT_IDX = [VAEP_FEATURES.index(c) for c in CATEGORICAL_FEATURES]
print(f"\nCategorical indices: {CAT_IDX}")

## 4. Training Loop

For each fold:
1. split the four training leagues into train / validation (85 / 15, by match) for early stopping;
2. train the scoring and conceding classifiers;
3. predict the held-out league;
4. compute each action's value per match;
5. save models and predictions immediately, so completed folds are skipped if the run restarts.

In [ ]:
# ── CatBoost parameters ───────────────────────────────────────────────────────
CB_PARAMS = dict(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    eval_metric="Logloss",
    cat_features=CAT_IDX,
    task_type="GPU" if HAS_GPU else "CPU",
    devices="0",
    verbose=0,
    early_stopping_rounds=50,
    random_seed=42,
    use_best_model=True,
)

In [ ]:
def compute_vaep_values_game(game_df, scores_col, concedes_col):
    """Per-action values within one match:  V(a_i) = dP_scores(a_i) - dP_concedes(a_i).

    - When possession changes, the previous state is read from the other team's side,
      so 'scores' and 'concedes' swap.
    - A goal ends the passage of play: the next action starts from a fresh baseline,
      like the first action of the match.
    """
    p_s  = game_df[scores_col].values
    p_c  = game_df[concedes_col].values
    tids = game_df["team_id"].values
    goal = game_df["is_goal_action"].values.astype(bool)
    n    = len(game_df)

    d_s = np.empty(n)
    d_c = np.empty(n)

    # First action: baseline is zero (pre-kick-off state)
    d_s[0] = p_s[0]
    d_c[0] = p_c[0]

    for i in range(1, n):
        if goal[i - 1]:                      # fresh baseline after a goal
            d_s[i] = p_s[i]
            d_c[i] = p_c[i]
        elif tids[i] == tids[i - 1]:         # same team
            d_s[i] = p_s[i] - p_s[i - 1]
            d_c[i] = p_c[i] - p_c[i - 1]
        else:                                # possession changed: swap perspective
            d_s[i] = p_s[i] - p_c[i - 1]
            d_c[i] = p_c[i] - p_s[i - 1]

    return d_s - d_c                         # more scoring chance + less conceding chance

In [ ]:
# ── Tracking containers ──────────────────────────────────────────────────────
all_eval_rows  = []          # evaluation metrics per fold
all_model_info = []          # best iterations and timing

# ── Check which folds are already done (resume support) ──────────────────────
completed_folds = set()
for league in LEAGUES:
    pred_path = PREDS_DIR / f"vaep_predictions_{league}.parquet"
    if pred_path.exists():
        completed_folds.add(league)

if completed_folds:
    print(f"✅ Already completed folds: {sorted(completed_folds)}")
    print(f"   Remaining: {sorted(set(LEAGUES) - completed_folds)}")
else:
    print("No completed folds found — starting from scratch.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LEAVE-ONE-LEAGUE-OUT LOOP
# ══════════════════════════════════════════════════════════════════════════════
total_start = time.time()

for fold_idx, (held_out, train_leagues) in enumerate(LOLO_FOLDS.items()):

    if held_out in completed_folds:
        print(f"FOLD {fold_idx+1}/5 — {held_out} — already saved, skipped")
        continue

    print(f"\n{'='*70}\nFOLD {fold_idx+1}/5 — held out: {held_out}   train: {train_leagues}\n{'='*70}")
    fold_t0 = time.time()

    # ── Masks: 15% of training matches are held back for early stopping ──────
    train_mask = df["league_country"].isin(train_leagues)
    test_mask  = df["league_country"] == held_out
    train_gids = df.loc[train_mask, "game_id"].unique()
    trn_gids, val_gids = train_test_split(train_gids, test_size=0.15, random_state=42)
    trn_mask = df["game_id"].isin(trn_gids) & train_mask
    val_mask = df["game_id"].isin(val_gids) & train_mask
    print(f"  Train: {trn_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}")

    X_trn_df  = df.loc[trn_mask,  VAEP_FEATURES]
    X_val_df  = df.loc[val_mask,  VAEP_FEATURES]
    X_test_df = df.loc[test_mask, VAEP_FEATURES]

    # ── Train the scoring and conceding classifiers ───────────────────────────
    models, preds = {}, {}
    for target in ("scores", "concedes"):
        y_col = f"y_{target}"
        t0 = time.time()
        m = CatBoostClassifier(**CB_PARAMS)
        m.fit(Pool(X_trn_df, df.loc[trn_mask, y_col].values, cat_features=CATEGORICAL_FEATURES),
              eval_set=Pool(X_val_df, df.loc[val_mask, y_col].values, cat_features=CATEGORICAL_FEATURES))
        elapsed = time.time() - t0

        mpath = MODELS_DIR / f"fold_{held_out}_{target}.cbm"
        m.save_model(str(mpath))
        models[target] = m
        preds[target]  = m.predict_proba(X_test_df)[:, 1]

        all_model_info.append({"fold": held_out, "target": target,
                               "best_iter": m.get_best_iteration(), "train_time_s": round(elapsed)})
        print(f"  P({target:<8s})  best_iter={m.get_best_iteration():>4d}  time={elapsed:.0f}s  → {mpath.name}")

        # ── Out-of-league evaluation ──────────────────────────────────────────
        y_test = df.loc[test_mask, y_col].values
        all_eval_rows.append({"fold": held_out, "model": target,
                              "brier": round(brier_score_loss(y_test, preds[target]), 6),
                              "roc_auc": round(roc_auc_score(y_test, preds[target]), 4),
                              "n": len(y_test), "pos_rate": round(y_test.mean(), 5)})

    # ── Action values, match by match ─────────────────────────────────────────
    df_test = df.loc[test_mask].copy()
    df_test["p_scores"]   = preds["scores"]
    df_test["p_concedes"] = preds["concedes"]
    values = np.zeros(len(df_test))
    for gid, idx in df_test.groupby("game_id", sort=False).indices.items():
        values[idx] = compute_vaep_values_game(df_test.iloc[idx], "p_scores", "p_concedes")
    df_test["vaep_value"] = values

    out_cols = ["game_id", "league_country", "team_id", "player_id",
                "absolute_second", "minute", "is_post_75",
                "type_id", "action_type", "result", "score_state",
                "p_scores", "p_concedes", "vaep_value",
                "y_scores", "y_concedes", "is_goal_action"]
    out_path = PREDS_DIR / f"vaep_predictions_{held_out}.parquet"
    df_test[out_cols].to_parquet(out_path, index=False)
    print(f"  Fold {held_out} done in {(time.time()-fold_t0)/60:.1f} min → {out_path.name}"
          f"   (mean value {values.mean():.6f}, std {values.std():.4f})")

    del X_trn_df, X_val_df, X_test_df, df_test, models, preds
    gc.collect()

print(f"\nAll folds complete — {(time.time()-total_start)/60:.1f} min")

## 5. Save Evaluation Metrics

In [ ]:
# ── Save evaluation table ─────────────────────────────────────────────────────
eval_df = pd.DataFrame(all_eval_rows)
eval_path = STAGE3_DIR / "evaluation_metrics.csv"
eval_df.to_csv(eval_path, index=False)
print(f"✅ Metrics saved → {eval_path}")
print(eval_df.to_string(index=False))

In [ ]:
# ── Save model training info ─────────────────────────────────────────────────
info_df = pd.DataFrame(all_model_info)
info_path = STAGE3_DIR / "model_training_info.csv"
info_df.to_csv(info_path, index=False)
print(f"\n✅ Training info saved → {info_path}")
print(info_df.to_string(index=False))

## 6. Combine Predictions Across Folds

Every action now has a value from a model that never saw its match.

In [ ]:
dfs = []
for league in LEAGUES:
    p = PREDS_DIR / f"vaep_predictions_{league}.parquet"
    dfs.append(pd.read_parquet(p))
    print(f"  Loaded {league}: {len(dfs[-1]):,} rows")

df_all = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

out_path = STAGE3_DIR / "vaep_all_leagues.parquet"
df_all.to_parquet(out_path, index=False)
print(f"\n✅ Combined file: {len(df_all):,} rows → {out_path}")
print(f"   Size: {out_path.stat().st_size/1e6:.1f} MB")

## 7. Out-of-League Performance

In [ ]:
eval_df = pd.read_csv(STAGE3_DIR / "evaluation_metrics.csv")
summary = (eval_df.groupby("model")
                  .agg(brier_mean=("brier", "mean"), brier_std=("brier", "std"),
                       auc_mean=("roc_auc", "mean"), auc_std=("roc_auc", "std"))
                  .round(5))
print("=== Mean ± std across the 5 held-out leagues ===")
print(summary.to_string())

In [ ]:
# ── Calibration: one panel per held-out league ────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for col_idx, league in enumerate(LEAGUES):
    pred_df = pd.read_parquet(PREDS_DIR / f"vaep_predictions_{league}.parquet")
    for row_idx, target in enumerate(("scores", "concedes")):
        ax = axes[row_idx, col_idx]
        prob_true, prob_pred = calibration_curve(pred_df[f"y_{target}"].values, pred_df[f"p_{target}"].values,
                                                 n_bins=10, strategy="quantile")
        ax.plot(prob_pred, prob_true, "o-", color="steelblue", markersize=4, linewidth=1.2)
        lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
        ax.plot([0, lim], [0, lim], "--", color="gray", alpha=0.5)
        ax.set_title(f"{league}\nP({target})", fontsize=9)
        if col_idx == 0:
            ax.set_ylabel("Empirical probability")
        if row_idx == 1:
            ax.set_xlabel("Predicted probability")
    del pred_df
    gc.collect()
plt.suptitle("Calibration on held-out leagues", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "calibration_all_folds.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Action Values

In [ ]:
df_all = pd.read_parquet(STAGE3_DIR / "vaep_all_leagues.parquet")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df_all["vaep_value"].clip(-0.15, 0.15), bins=120, color="steelblue", alpha=0.8, edgecolor="none")
ax.axvline(0, color="gray", ls="--")
ax.set_xlabel("Action value")
ax.set_ylabel("Actions")
ax.set_title("Action values, all leagues")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "vaep_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"mean = {df_all['vaep_value'].mean():.6f}   std = {df_all['vaep_value'].std():.4f}")

In [ ]:
by_type = (df_all.groupby("action_type")
                 .agg(mean_value=("vaep_value", "mean"), count=("vaep_value", "size"))
                 .sort_values("mean_value", ascending=False))
print("=== Mean action value by action type ===")
print(by_type.round(6).to_string())

by_league = df_all.groupby("league_country")["vaep_value"].agg(["size", "mean", "std"]).round(6)
print("\n=== Action value by league ===")
print(by_league.to_string())

**Next → Stage 4** (in-game stakes). Stage 5 joins `vaep_all_leagues.parquet` with the Stage 4 output.